# [6.4] Crosscoders and Model Diffing - Exercises

Implement the local crosscoder and model-diffing primitives. The CUDA report uses paired `gelu-1l` and `solu-1l` TransformerLens checkpoints, but these exercises are CPU-safe and focus on the exact contracts used by the real-model path.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import torch as t
import torch.nn.functional as F

chapter = "chapter6_sparse_feature_methods"
section = "part4_crosscoders_model_diffing"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part4_crosscoders_model_diffing.tests as tests

FeatureOwner = Literal["shared", "model_a", "model_b"]


@dataclass(frozen=True)
class CrosscoderOutput:
    shared_acts: t.Tensor
    model_a_specific_acts: t.Tensor
    model_b_specific_acts: t.Tensor
    reconstructed_model_a: t.Tensor
    reconstructed_model_b: t.Tensor


@dataclass(frozen=True)
class CrosscoderReconstructionReport:
    model_a_mse: float
    model_b_mse: float
    shared_active_fraction: float
    model_a_passes: bool
    model_b_passes: bool
    shared_reconstructs_both: bool


@dataclass(frozen=True)
class FeatureSpecificityReport:
    feature_id: int
    model_a_mean: float
    model_b_mean: float
    specificity: float
    owner: FeatureOwner


@dataclass(frozen=True)
class BehaviorDeltaPredictionReport:
    feature_id: int
    auc: float
    positive_mean: float
    negative_mean: float
    passes_threshold: bool


@dataclass(frozen=True)
class CrosscoderAblationReport:
    baseline_delta: float
    ablated_delta: float
    random_ablated_delta: float
    delta_reduction: float
    random_reduction: float
    passes_control: bool


## Crosscoder Reconstruction

Decode shared features into both model spaces, and decode model-specific features only into their matching model space.


In [ ]:
def decode_crosscoder(
    shared_acts: t.Tensor,
    model_a_specific_acts: t.Tensor,
    model_b_specific_acts: t.Tensor,
    shared_decoder_a: t.Tensor,
    shared_decoder_b: t.Tensor,
    model_a_decoder: t.Tensor,
    model_b_decoder: t.Tensor,
) -> CrosscoderOutput:
    raise NotImplementedError()


def crosscoder_reconstruction_report(
    model_a_activations: t.Tensor,
    model_b_activations: t.Tensor,
    output: CrosscoderOutput,
    *,
    mse_threshold: float = 1e-6,
) -> CrosscoderReconstructionReport:
    raise NotImplementedError()


tests.test_decode_crosscoder_reconstructs_shared_and_specific_spaces(
    decode_crosscoder,
    crosscoder_reconstruction_report,
)


## Feature Specificity

Classify each feature as shared, model-A-specific, or model-B-specific from the sign and magnitude of its cross-model mean activation difference.


In [ ]:
def feature_specificity_report(
    model_a_feature_acts: t.Tensor,
    model_b_feature_acts: t.Tensor,
    feature_id: int,
    *,
    shared_threshold: float = 0.1,
) -> FeatureSpecificityReport:
    raise NotImplementedError()


def classify_features_by_specificity(
    model_a_feature_acts: t.Tensor,
    model_b_feature_acts: t.Tensor,
    *,
    shared_threshold: float = 0.1,
) -> list[FeatureOwner]:
    raise NotImplementedError()


tests.test_feature_specificity_classifies_shared_model_a_and_model_b(
    feature_specificity_report,
    classify_features_by_specificity,
)


## Behavior-Delta Prediction

Use signed AUC to test whether a feature predicts paired examples where model B differs from model A.


In [ ]:
def roc_auc_binary(scores: t.Tensor, labels: t.Tensor) -> float:
    raise NotImplementedError()


def behavior_delta_prediction_report(
    feature_scores: t.Tensor,
    behavior_delta_labels: t.Tensor,
    *,
    feature_id: int,
    min_auc: float = 0.8,
) -> BehaviorDeltaPredictionReport:
    raise NotImplementedError()


tests.test_behavior_delta_prediction_uses_signed_auc_and_means(
    behavior_delta_prediction_report,
    roc_auc_binary,
)


## Paired Behavior Deltas

The section convention is model B minus model A. Keep the sign until you explicitly report absolute reductions.


In [ ]:
def toy_behavior_delta_scores(
    model_a_scores: t.Tensor,
    model_b_scores: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_toy_behavior_delta_scores_are_model_b_minus_model_a(
    toy_behavior_delta_scores,
)


## Ablation Controls

A model-specific feature needs a causal control: removing it should reduce the behavior delta more than removing a random matched feature.


In [ ]:
def crosscoder_ablation_report(
    baseline_behavior_deltas: t.Tensor,
    ablated_behavior_deltas: t.Tensor,
    random_ablated_behavior_deltas: t.Tensor,
) -> CrosscoderAblationReport:
    raise NotImplementedError()


tests.test_crosscoder_ablation_requires_target_to_beat_random_control(
    crosscoder_ablation_report,
)


## Full Verification

After filling in the notebook, compare your implementation to `solutions.py`. The full CUDA path is run separately by `solutions.run_gpu_test(max_vram_gb=24.0)` and should report paired-model reconstruction, behavior-delta AUC, top-direction ablation, random-direction control, and peak VRAM.


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
